In [1]:
import os 
import pwd
import json
import pandas as pd
import numpy as np
import regex as re

from sqlalchemy import create_engine
import sqlalchemy as sa
from sqlalchemy.engine import URL
from sqlalchemy import text

from IPython.display import display, HTML
import statistics
from sklearn.impute import SimpleImputer

# import xgboost
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pyodbc
import pandas as pd

# fn to setup SQL connectivity
def get_sql_connection():
    # omitted for security purposes
    pass
	
def send_query(sql_query):

    cursor.execute(sql_query)

    # Fetch column names
    columns = [column[0] for column in cursor.description]

    # Fetch all rows
    rows = cursor.fetchall()

    # Ensure rows are converted properly (each row is a tuple)
    df = pd.DataFrame.from_records([tuple(row) for row in rows], columns=columns)

    return df

## Main ##

#get a connection to mssql on HHCMITSQL001
conn = get_sql_connection()
cursor = conn.cursor()


## Extract Blood Cultures

In [ ]:
df = send_query("select * from OMITTED_FOR_SECURITY_PURPOSES")

In [5]:
df['OrderedYear'] = df['OrderedDateKey'].astype(str).str[:4].astype(int)
df['OrderedYear'].value_counts().sort_index() # We restrict to post-2021 due to system updates, differences in historical data

OrderedYear
2021    289773
2022    180044
2023    290369
2024    249265
2025    123563
Name: count, dtype: int64

In [26]:
df = df.loc[(df['OrderedYear']>2021) & (df['IsFinal']==1)]
df.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/base_blood_cultures.csv", index = False)

### Define Positive Blood Cultures

In [16]:
labels = pd.read_csv("/datafs_a/carolgao/haim14_data/blood_infection/BSI_values_to_labels_2022-25.csv")
labels = labels.rename(columns = {"culture status (P = positive, N = negative, N/A = does not address culture status)": "culture_status"})

In [22]:
df = pd.merge(df, labels[['Value', 'culture_status', 'Potential Contaminant']], how = 'left', on = 'Value') 

In [32]:
df['BloodOrganism'] = 0 
df['culture_status'] = df['culture_status'].str.lower()
# drop unclear components 
df = df.loc[(df['culture_status'] != "n/a") & (df['culture_status'] != "n/a ") & (df['culture_status'] != "?") & (df['culture_status'] != "?n/a")]
# Code positivs 
df.loc[df['culture_status'] == "p", 'BloodOrganism'] = 1
df.loc[df['culture_status'].isnull() == True, 'culture_status'] = ""
df.loc[df['culture_status'].str.contains('positive'), 'BloodOrganism'] = 1

In [33]:
len(df['LabOrderEpicId'].unique())

357212

In [112]:
# List all the blood cultures ordered for a unique time during an encounter
df["CollectionInstant"] = pd.to_datetime(df["CollectionInstant"])
# Truncate to minutes (removes seconds)
df["CollectionInstant_Rounded"] = df["CollectionInstant"].dt.floor("T") 
blood_sets = df[['EncounterKey','OrderedDateKey', 'CollectionInstant_Rounded', 'LabOrderEpicId']].drop_duplicates()
blood_sets_unique = blood_sets.groupby(['EncounterKey','OrderedDateKey', 'CollectionInstant_Rounded'], as_index=False).agg(list)

In [113]:
# Code number of sets drawn
blood_sets_unique['Sets'] = blood_sets_unique['LabOrderEpicId'].apply(lambda x: len(x))
blood_sets_unique= blood_sets_unique.rename(columns = {'LabOrderEpicId': 'LabOrderEpicId_Sets'})
blood_sets_unique['LabOrderEpicId_Sets'] = blood_sets_unique['LabOrderEpicId_Sets'].astype(str)

In [115]:
# Merge with organism info 
blood_sets_unique = pd.merge(df, blood_sets_unique, how="inner", on=['EncounterKey', 'OrderedDateKey', 'CollectionInstant_Rounded'])

In [116]:
# Both positive - take max by laborderepicid_x, then min by encounter 
blood_sets_unique['Set_Positive1'] = blood_sets_unique.groupby(['LabOrderEpicId_Sets', 'OrderedDateKey', 'CollectionInstant_Rounded'])['HasOrganism'].transform(max)
blood_sets_unique['Set_Positive2'] = blood_sets_unique.groupby(['LabOrderEpicId_Sets', 'OrderedDateKey', 'CollectionInstant_Rounded'])['BloodOrganism'].transform(max)

blood_sets_unique['Set_Positive'] = blood_sets_unique[["Set_Positive1", "Set_Positive2"]].max(axis=1)
blood_sets_unique['BothPositive'] = blood_sets_unique.groupby(['EncounterKey', 'OrderedDateKey', 'CollectionInstant_Rounded'])['Set_Positive'].transform(min)

In [117]:
# Any positive 
blood_sets_unique['Positive1'] = blood_sets_unique.groupby(['EncounterKey', 'OrderedDateKey', 'CollectionInstant_Rounded'])['HasOrganism'].transform(max)
blood_sets_unique['Positive2'] = blood_sets_unique.groupby(['EncounterKey', 'OrderedDateKey', 'CollectionInstant_Rounded'])['BloodOrganism'].transform(max)

blood_sets_unique['Positive'] = blood_sets_unique[["Positive1", "Positive2"]].max(axis=1)

In [119]:
# False positive 
blood_sets_unique['FalsePositive'] = 0 
false_keywords = ['staphylococcus','epidermidis','auricularis','carnosus','condiment','debuckii','massiliensis',
                  'piscifermentans','simulans','capitis','caprae','saccharolyticus', 'borealis','devriesei','haemolyticus',
                  'hominis','agnetis','chromogenes','cornubiensis','felis','microti','muscae','rostri','arlettae','caeli',
                  'cohnii','equorum','gallinarum','kloosii','leei','nepalensis','saprophyticus','succinus','xylosus',
                  'fleurettii','lentus','sciuri','stepanovicii','vitulinus','simulans','pasteuri','warneri',
                  'Coagulase negative Staphylococcus','Bacillus species','Micrococcus','Cutibacterium acnes',
                  'Corynebacterium species']
for word in false_keywords: 
    blood_sets_unique['FalseKeywords'] = (blood_sets_unique['Value'].str.contains(word,case=False, na=False)).astype(int)

blood_sets_unique['FalsePositive'] = blood_sets_unique.groupby(['EncounterKey', 'OrderedDateKey', 'CollectionInstant_Rounded'])['FalseKeywords'].transform(max)

In [120]:
blood_sets_unique.loc[(blood_sets_unique['BloodOrganism'] == 1) & (blood_sets_unique['HasOrganism'] == 0)];

In [121]:
all_cultures_unique = blood_sets_unique[['PatientDurableKey', 'primarymrn', 'EncounterKey', 'OrderedYear', 'OrderedDateKey', 'CollectionInstant_Rounded', 'Sets', 'Positive', 'BothPositive', 'FalsePositive']].drop_duplicates()

In [122]:
# Correct for false positives 
all_cultures_unique.loc[all_cultures_unique['FalsePositive'] == 1, 'Positive'] = 0 

In [123]:
all_cultures_unique = all_cultures_unique.sort_values(by = ['OrderedDateKey','CollectionInstant_Rounded'], ascending = True)
# Onky keep first encounter
all_cultures_enc = all_cultures_unique.drop_duplicates(subset='EncounterKey', keep='first',ignore_index=True)

In [124]:
all_cultures_enc.groupby(['OrderedYear']).agg(
    positive_rate=('Positive', 'mean'),
    N = ('Positive', 'count')
)

,positive_rate,N
OrderedYear,,
2022,0.087850,44963
2023,0.086243,45766
2024,0.091937,39603
2025,0.094065,20624


In [125]:
all_cultures_enc.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/all_blood_cultures_unique_encounter.csv", index = False)

In [61]:
two_sets_unique = all_cultures_unique.loc[all_cultures_unique['Sets'] == 2]
print("Positive:", all_cultures_unique['Positive'].mean())
print("Both Positive:", all_cultures_unique['BothPositive'].mean())

Positive: 0.09214443355987635
Both Positive: 0.07838413716080488


In [62]:
two_sets_unique = all_cultures_unique.loc[all_cultures_unique['Sets'] == 2]
print(len(two_sets_unique['Positive']))
print(two_sets_unique['Positive'].mean())
print(two_sets_unique['BothPositive'].mean())

82835
0.11172813424277178
0.09652924488440877


In [63]:
two_sets_unique = two_sets_unique.sort_values(by = ['OrderedDateKey','CollectionInstant_Rounded'], ascending = True)
# Onky keep first encounter
two_sets_enc = two_sets_unique.drop_duplicates(subset='EncounterKey', keep='first',ignore_index=True)

In [34]:
two_sets_enc.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/two_sets_blood_cultures.csv", index = False)

In [64]:
len(two_sets_enc)

75383

In [65]:
two_sets_enc.groupby(['OrderedYear']).agg(
    positive_rate=('Positive', 'mean'),
    N = ('Positive', 'count')
)

,positive_rate,N
OrderedYear,,
2022,0.100174,23589
2023,0.095383,23715
2024,0.108059,19619
2025,0.151064,8460


In [58]:
df = df.sort_values(by = ['OrderedDateKey','CollectionInstant_Rounded'], ascending = True)
df = df.drop_duplicates(subset='EncounterKey', keep='first',ignore_index=True)

In [140]:
len(df['EncounterKey'].unique())

150959

In [128]:
len(all_cultures_enc['EncounterKey'].unique())

150956

## Extract Features

In [129]:
# base = pd.read_csv('/datafs_a/haim14_data/blood_infection/two_sets_blood_cultures.csv')
encounters =",".join([str(x) for x in df['EncounterKey'].unique()])

### Get vitals

In [132]:
vital_types = send_query(f"select distinct d2.FlowsheetRowKey, d2.DisplayName, d2.ValueType from CAB_EPIC.dbo.FlowsheetRowDim d2 \
    where d2.FlowsheetRowKey > 0 and (d2.DisplayName = 'SpO2' or d2.DisplayName = 'Temp' or d2.DisplayName = 'Pulse' or d2.DisplayName = 'Resp')")
vital_keys = ",".join([str(x) for x in vital_types['FlowsheetRowKey'].unique()])


In [133]:
%%time
vitals_raw = send_query(f"select distinct d1.PatientDurableKey, d1.EncounterKey, d1.FlowsheetRowKey, d1.DateKey, d1.TimeOfDayKey, \
d1.Value, d1.NumericValue from CAB_EPIC.dbo.FlowsheetValueFact d1 \
where d1.FlowsheetRowKey in ({vital_keys}) and d1.EncounterKey in ({encounters})")

CPU times: user 2min 28s, sys: 11 s, total: 2min 39s
Wall time: 10min 48s


In [141]:
vitals_raw = pd.merge(vitals_raw, vital_types, on = ['FlowsheetRowKey'], how = 'left')

In [150]:
vitals_raw.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/vitals_raw_all.csv", index = False)

### Get pulse

In [151]:
pulse_types = send_query(f"select distinct d2.FlowsheetRowKey, d2.DisplayName, d2.ValueType from CAB_EPIC.dbo.FlowsheetRowDim d2 \
    where d2.FlowsheetRowKey > 0 and (d2.DisplayName = 'heart rate' or d2.ValueType = 'Pulse (Heart Rate)')")
pulse_keys = ",".join([str(x) for x in pulse_types['FlowsheetRowKey'].unique()])

In [152]:
%%time
pulse_raw = send_query(f"select distinct d1.PatientDurableKey, d1.EncounterKey, d1.FlowsheetRowKey, d1.DateKey, d1.TimeOfDayKey, \
d1.Value, d1.NumericValue from CAB_EPIC.dbo.FlowsheetValueFact d1 \
where d1.FlowsheetRowKey in ({pulse_keys}) and d1.EncounterKey in ({encounters})")
pulse_raw = pd.merge(pulse_raw, pulse_types, on = ['FlowsheetRowKey'], how = 'left')
pulse_raw.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/pulse_raw_all.csv", index = False)

CPU times: user 29.7 ms, sys: 5.91 ms, total: 35.6 ms
Wall time: 7min 29s


### Get SPO2

In [153]:
spo2_types = send_query(f"select distinct d2.FlowsheetRowKey, d2.DisplayName, d2.ValueType from CAB_EPIC.dbo.FlowsheetRowDim d2 \
    where d2.FlowsheetRowKey > 0 and (d2.DisplayName like '%SpO2%' and d2.DisplayName != 'SpO2')")
spo2_keys = ",".join([str(x) for x in spo2_types['FlowsheetRowKey'].unique()])

In [154]:
%%time
spo2_raw = send_query(f"select distinct d1.PatientDurableKey, d1.EncounterKey, d1.FlowsheetRowKey, d1.DateKey, d1.TimeOfDayKey, \
d1.Value, d1.NumericValue from CAB_EPIC.dbo.FlowsheetValueFact d1 \
where d1.FlowsheetRowKey in ({spo2_keys}) and d1.EncounterKey in ({encounters})")
spo2_raw = pd.merge(spo2_raw, spo2_types, on = 'FlowsheetRowKey', how = 'left')
spo2_raw.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/spo2_raw_all.csv", index = False)

CPU times: user 549 ms, sys: 23.8 ms, total: 572 ms
Wall time: 7min 37s


In [155]:
spo2_raw.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/spo2_raw.csv", index = False)